# Hands-On Pertemuan 14: Advanced Machine Learning using Spark MLlib

## Objectives:
- Understand and implement advanced machine learning tasks using Spark MLlib.
- Build and evaluate models using real-world datasets.
- Explore techniques like feature engineering and hyperparameter tuning.


## Introduction to Spark MLlib
Spark MLlib is a scalable library for machine learning that integrates seamlessly with the Spark ecosystem. It supports a wide range of tasks, including regression, classification, clustering, and collaborative filtering.

In [4]:
# Example: Linear Regression with Spark MLlib
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

# Initialize Spark Session
spark = SparkSession.builder.appName('MLlib Example').getOrCreate()

# Load sample data
data = [(1, 5.0, 20.0), (2, 10.0, 25.0), (3, 15.0, 30.0), (4, 20.0, 35.0)]
columns = ['ID', 'Feature', 'Target']
df = spark.createDataFrame(data, columns)

# Prepare data for modeling
assembler = VectorAssembler(inputCols=['Feature'], outputCol='Features')
df_transformed = assembler.transform(df)

# Train a linear regression model
lr = LinearRegression(featuresCol='Features', labelCol='Target')
model = lr.fit(df_transformed)

# Print model coefficients
print(f'Coefficients: {model.coefficients}')
print(f'Intercept: {model.intercept}')


ModuleNotFoundError: No module named 'pyspark'

In [9]:
# Practice: Logistic Regression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler

# Example dataset
data = [(1, 2.0, 3.0, 0), (2, 1.0, 5.0, 1), (3, 2.5, 4.5, 1), (4, 3.0, 6.0, 0)]
columns = ['ID', 'Feature1', 'Feature2', 'Label']
df = spark.createDataFrame(data, columns)

# Convert Features column to VectorUDT type
assembler = VectorAssembler(inputCols=['Feature1', 'Feature2'], outputCol='FeaturesVec')
df_transformed = assembler.transform(df)

# Train logistic regression model
lr = LogisticRegression(featuresCol='FeaturesVec', labelCol='Label')
model = lr.fit(df_transformed)

# Display coefficients and summary
print(f'Coefficients: {model.coefficients}')
print(f'Intercept: {model.intercept}')


Coefficients: [-12.262057963810907,4.087352278029272]
Intercept: 11.568912761288276


In [10]:
# Practice: KMeans Clustering
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

# Example dataset
data = [(1, 1.0, 1.0), (2, 5.0, 5.0), (3, 10.0, 10.0), (4, 15.0, 15.0)]
columns = ['ID', 'Feature1', 'Feature2']
df = spark.createDataFrame(data, columns)

# Convert Features column to VectorUDT type
assembler = VectorAssembler(inputCols=['Feature1', 'Feature2'], outputCol='FeaturesVec')
df_transformed = assembler.transform(df)

# Train KMeans clustering model
kmeans = KMeans(featuresCol='FeaturesVec', k=2)
model = kmeans.fit(df_transformed)

# Show cluster centers
centers = model.clusterCenters()
print(f'Cluster Centers: {centers}')


Cluster Centers: [array([12.5, 12.5]), array([3., 3.])]


## Homework
- Load a real-world dataset into Spark and prepare it for machine learning tasks.
- Build a classification model using Spark MLlib and evaluate its performance.
- Explore hyperparameter tuning using cross-validation.


In [6]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Initialize Spark session
spark = SparkSession.builder.appName('MLlib Example').getOrCreate()

# Load dataset
df = spark.read.csv('amu.us.txt', header=True, inferSchema=True)

# Prepare data for machine learning
assembler = VectorAssembler(inputCols=['Open', 'High', 'Low', 'Close', 'Volume'], outputCol='features', handleInvalid='skip')
df_transformed = assembler.transform(df)

# Add a label column (for demonstration purposes, let's assume 'Close' > 20 is class 1, else class 0)
df_transformed = df_transformed.withColumn('label', (df_transformed['Close'] > 20).cast('integer'))

# Split data into training and test sets
train_data, test_data = df_transformed.randomSplit([0.8, 0.2], seed=1234)

# Initialize and train the classification model
lr = LogisticRegression(featuresCol='features', labelCol='label')
lr_model = lr.fit(train_data)

# Evaluate the model
predictions = lr_model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol='label')
accuracy = evaluator.evaluate(predictions)
print(f'Accuracy: {accuracy}')

# Hyperparameter tuning using cross-validation
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.1, 0.01]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

cv_model = crossval.fit(train_data)
cv_predictions = cv_model.transform(test_data)
cv_accuracy = evaluator.evaluate(cv_predictions)
print(f'Cross-validated Accuracy: {cv_accuracy}')

Accuracy: 1.0
Cross-validated Accuracy: 1.0


In [12]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Initialize Spark session
spark = SparkSession.builder.appName('MLlib Example').getOrCreate()

# Load dataset
df = spark.read.csv('amu.us.txt', header=True, inferSchema=True)

# Prepare data for machine learning
assembler = VectorAssembler(inputCols=['Open', 'High', 'Low', 'Close', 'Volume'], outputCol='features')
df_transformed = assembler.transform(df)

# Add a label column (for demonstration purposes, let's assume 'Close' > 20 is class 1, else class 0)
df_transformed = df_transformed.withColumn('label', (df_transformed['Close'] > 20).cast('integer'))

# Split data into training and test sets
train_data, test_data = df_transformed.randomSplit([0.8, 0.2], seed=1234)

# Initialize and train the classification model
lr = LogisticRegression(featuresCol='features', labelCol='label')
lr_model = lr.fit(train_data)

# Evaluate the model
predictions = lr_model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol='label')
accuracy = evaluator.evaluate(predictions)
print(f'Accuracy: {accuracy}')

# Hyperparameter tuning using cross-validation
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.1, 0.01]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

cv_model = crossval.fit(train_data)
cv_predictions = cv_model.transform(test_data)
cv_accuracy = evaluator.evaluate(cv_predictions)
print(f'Cross-validated Accuracy: {cv_accuracy}')

Accuracy: 1.0


Cross-validated Accuracy: 1.0
